In [ ]:
import os
import sys

project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.append(project_root)

import torch
import torch.nn as nn
import torch.optim as optim

from src.data_utils import get_mnist_dataloaders, get_class_names
from src.models import MLPBaseline, SimpleCNN
from src.train import fit_model
from src.evaluate import collect_predictions, compute_classification_metrics
from src.visualize import (
    plot_training_history,
    plot_confusion_matrix,
    show_sample_batch,
    visualize_conv1_kernels,
    visualize_feature_maps,
    show_misclassified_examples,
)

In [ ]:
SEED = 42
BATCH_SIZE = 64
EPOCHS = 10
LR = 1e-3

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE:", DEVICE)

In [ ]:
train_loader, val_loader, test_loader = get_mnist_dataloaders(
    data_dir="../data",
    batch_size=BATCH_SIZE,
    val_size=5000,
    num_workers=0,
    augment=False,
    normalize=False,
    seed=SEED,
)

class_names = get_class_names()
show_sample_batch(train_loader, class_names=class_names, n=16)

In [ ]:
torch.manual_seed(SEED)

mlp = MLPBaseline().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp.parameters(), lr=LR)

mlp_result = fit_model(
    model=mlp,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=DEVICE,
    epochs=EPOCHS,
)

In [ ]:
mlp_logits, mlp_preds, mlp_labels = collect_predictions(mlp_result["model"], test_loader, DEVICE)
mlp_metrics = compute_classification_metrics(mlp_labels, mlp_preds, class_names=class_names)

print("MLP Test Metrics")
print(f"Accuracy        : {mlp_metrics['accuracy']:.4f}")
print(f"Macro Precision : {mlp_metrics['macro_precision']:.4f}")
print(f"Macro Recall    : {mlp_metrics['macro_recall']:.4f}")
print(f"Macro F1        : {mlp_metrics['macro_f1']:.4f}")
print()
print(mlp_metrics["classification_report"])

In [ ]:
torch.manual_seed(SEED)

cnn = SimpleCNN().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(cnn.parameters(), lr=LR)

cnn_result = fit_model(
    model=cnn,
    train_loader=train_loader,
    val_loader=val_loader,
    criterion=criterion,
    optimizer=optimizer,
    device=DEVICE,
    epochs=EPOCHS,
)

In [ ]:
cnn_logits, cnn_preds, cnn_labels = collect_predictions(cnn_result["model"], test_loader, DEVICE)
cnn_metrics = compute_classification_metrics(cnn_labels, cnn_preds, class_names=class_names)

print("CNN Test Metrics")
print(f"Accuracy        : {cnn_metrics['accuracy']:.4f}")
print(f"Macro Precision : {cnn_metrics['macro_precision']:.4f}")
print(f"Macro Recall    : {cnn_metrics['macro_recall']:.4f}")
print(f"Macro F1        : {cnn_metrics['macro_f1']:.4f}")
print()
print(cnn_metrics["classification_report"])

In [ ]:
plot_confusion_matrix(
    cnn_metrics["confusion_matrix"],
    class_names=class_names,
    save_path="../results/figures/g1_cnn_confusion_matrix.png",
)

In [ ]:
visualize_conv1_kernels(
    cnn_result["model"],
    save_path="../results/figures/g1_cnn_conv1_kernels.png",
)

In [ ]:
images, labels = next(iter(test_loader))
sample_idx = 0

visualize_feature_maps(
    cnn_result["model"],
    images[sample_idx],
    device=DEVICE,
    layer_name="conv1",
    max_channels=8,
    save_path="../results/figures/g1_feature_maps_conv1.png",
)

visualize_feature_maps(
    cnn_result["model"],
    images[sample_idx],
    device=DEVICE,
    layer_name="conv2",
    max_channels=8,
    save_path="../results/figures/g1_feature_maps_conv2.png",
)

In [ ]:
import numpy as np
import pandas as pd

cm = cnn_metrics["confusion_matrix"]
class_names = [str(i) for i in range(10)]

rows = []
for i in range(10):
    total = cm[i].sum()
    correct = cm[i, i]
    errors = total - correct
    row = {
        "class": i,
        "support": total,
        "correct": correct,
        "errors": errors,
        "recall": correct / total,
    }
    wrong_targets = []
    for j in range(10):
        if i != j and cm[i, j] > 0:
            wrong_targets.append(f"{j}:{cm[i,j]}")
    row["misclassified_as"] = ", ".join(wrong_targets)
    rows.append(row)

df_error = pd.DataFrame(rows).sort_values("errors", ascending=False)
df_error

In [ ]:
import os

save_path = "../results/logs/g1_best_cnn.pt"
os.makedirs(os.path.dirname(save_path), exist_ok=True)

torch.save(cnn_result["model"].state_dict(), save_path)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

save_dir = "../results/figures"
os.makedirs(save_dir, exist_ok=True)

epochs_mlp = np.arange(1, len(mlp_result["history"]["train_loss"]) + 1)
epochs_cnn = np.arange(1, len(cnn_result["history"]["train_loss"]) + 1)

plt.figure(figsize=(10, 5))

plt.plot(
    epochs_mlp,
    mlp_result["history"]["train_loss"],
    marker="o",
    label="MLP Train Loss"
)
plt.plot(
    epochs_mlp,
    mlp_result["history"]["val_loss"],
    marker="o",
    linestyle="--",
    label="MLP Val Loss"
)

plt.plot(
    epochs_cnn,
    cnn_result["history"]["train_loss"],
    marker="s",
    label="CNN Train Loss"
)
plt.plot(
    epochs_cnn,
    cnn_result["history"]["val_loss"],
    marker="s",
    linestyle="--",
    label="CNN Val Loss"
)

plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("MLP vs CNN Loss Curves")
plt.legend()
plt.tight_layout()

plt.savefig(f"{save_dir}/g1_mlp_cnn_loss_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

In [ ]:
import os
import pandas as pd

table_dir = "../results/tables"
os.makedirs(table_dir, exist_ok=True)

results_df = pd.DataFrame([
    {
        "model": "MLP",
        "best_val_acc": max(mlp_result["history"]["val_acc"]),
        "test_accuracy": mlp_metrics["accuracy"],
        "macro_precision": mlp_metrics["macro_precision"],
        "macro_recall": mlp_metrics["macro_recall"],
        "macro_f1": mlp_metrics["macro_f1"],
        "final_train_loss": mlp_result["history"]["train_loss"][-1],
        "final_val_loss": mlp_result["history"]["val_loss"][-1],
        "final_train_acc": mlp_result["history"]["train_acc"][-1],
        "final_val_acc": mlp_result["history"]["val_acc"][-1],
    },
    {
        "model": "CNN",
        "best_val_acc": max(cnn_result["history"]["val_acc"]),
        "test_accuracy": cnn_metrics["accuracy"],
        "macro_precision": cnn_metrics["macro_precision"],
        "macro_recall": cnn_metrics["macro_recall"],
        "macro_f1": cnn_metrics["macro_f1"],
        "final_train_loss": cnn_result["history"]["train_loss"][-1],
        "final_val_loss": cnn_result["history"]["val_loss"][-1],
        "final_train_acc": cnn_result["history"]["train_acc"][-1],
        "final_val_acc": cnn_result["history"]["val_acc"][-1],
    },
])

display(results_df)

results_df.to_csv(f"{table_dir}/g1_mlp_vs_cnn_summary.csv", index=False, encoding="utf-8-sig")

with open(f"{table_dir}/g1_mlp_vs_cnn_summary.md", "w", encoding="utf-8") as f:
    f.write(results_df.to_markdown(index=False))

print("saved to:", f"{table_dir}/g1_mlp_vs_cnn_summary.csv")
print("saved to:", f"{table_dir}/g1_mlp_vs_cnn_summary.md")